# 💻 Chapter 22: Debugging Probabilistic Systems
*Track 2 — Developers | Tier 2 Finale*

> **🎬 Hook:** How do you write a unit test for code that's supposed to be random? The answer: test the distribution, not the output.

**🎯 Objectives:** Test stochastic code statistically; use KS test, chi-square, and property-based testing for probabilistic systems.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# ── The Core Problem: Non-Deterministic Code ──
def naive_test():
    # WRONG: this test will randomly fail.
    rng2 = np.random.default_rng()
    result = rng2.integers(0, 10)
    assert result == 5, f"Expected 5, got {result}"

print("Testing probabilistic code correctly:")
print("Wrong: assert result == expected_value")
print("Right: assert distribution matches expected distribution")
print()

# ── Statistical Tests for Random Number Generators ──
def test_uniformity(samples, low, high, alpha=0.05):
    ks_stat, p_value = stats.kstest(samples, 'uniform',
                                     args=(low, high-low))
    return {'test': 'KS-Uniformity', 'stat': ks_stat, 'p': p_value,
            'pass': p_value > alpha}

def test_normality(samples, expected_mean, expected_std, alpha=0.05):
    z_scores = (samples - expected_mean) / expected_std
    ks_stat, p_value = stats.kstest(z_scores, 'norm')
    return {'test': 'KS-Normality', 'stat': ks_stat, 'p': p_value,
            'pass': p_value > alpha}

def test_categorical(samples, expected_probs, n_categories, alpha=0.05):
    counts = np.bincount(samples, minlength=n_categories)
    expected_counts = np.array(expected_probs) * len(samples)
    chi2, p_value = stats.chisquare(counts, expected_counts)
    return {'test': 'Chi2-Categorical', 'stat': chi2, 'p': p_value,
            'pass': p_value > alpha}

# Run tests
n = 10000
tests = [
    test_uniformity(rng.uniform(0, 1, n), 0, 1),
    test_normality(rng.normal(5, 2, n), 5, 2),
    test_categorical(rng.choice(4, n, p=[0.1,0.2,0.3,0.4]),
                    [0.1,0.2,0.3,0.4], 4),
]

print(f"{'Test':<20} {'Statistic':>12} {'p-value':>10} {'Pass?':>8}")
print("-" * 54)
for t in tests:
    status = "✅ PASS" if t['pass'] else "❌ FAIL"
    print(f"  {t['test']:<18} {t['stat']:>12.4f} {t['p']:>10.4f} {status:>8}")

In [ ]:
# ── Detect Biased Sampling ──
def test_random_function(sample_fn, n=10000, alpha=0.05):
    # Test if a function produces fair random samples.
    samples = [sample_fn() for _ in range(n)]
    unique_vals = sorted(set(samples))
    expected_p = 1 / len(unique_vals)
    counts = [samples.count(v) for v in unique_vals]
    expected = [n * expected_p] * len(unique_vals)
    chi2, p = stats.chisquare(counts, expected)
    return p > alpha, p, chi2

# Good sampler
def good_sampler():
    return rng.integers(0, 6)

# Bad sampler (biased! modulo bias)
def bad_sampler():
    return int(rng.integers(0, 2**31)) % 6  # slight modulo bias

# Test both
good_pass, good_p, _ = test_random_function(good_sampler)
bad_pass,  bad_p,  _ = test_random_function(bad_sampler)

print(f"Good sampler: p={good_p:.4f} → {'✅ PASS (uniform)' if good_pass else '❌ FAIL'}")
print(f"Bad sampler:  p={bad_p:.4f} → {'✅ PASS' if bad_pass else '❌ FAIL (biased!)'}")
print()

# Visualize the difference
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, fn, title in [(axes[0], good_sampler, 'Good Sampler'),
                       (axes[1], bad_sampler, 'Bad Sampler (modulo bias)')]:
    samples = [fn() for _ in range(100000)]
    counts = [samples.count(i) for i in range(6)]
    colors = ['#27ae60' if abs(c/100000 - 1/6) < 0.002 else '#e74c3c' for c in counts]
    ax.bar(range(6), [c/1000 for c in counts], color=colors, edgecolor='white')
    ax.axhline(100/6, color='red', lw=2, linestyle='--', label='Expected (uniform)')
    ax.set_title(f'{title}', fontweight='bold')
    ax.set_xlabel('Value'); ax.set_ylabel('Count (thousands)')
    ax.legend()

plt.tight_layout()
plt.savefig('ch22_sampling_test.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Property-Based Testing for Probabilistic Code ──
# Test PROPERTIES of distributions, not specific values

def verify_distribution_properties(samples, name=""):
    # Test key statistical properties of a sample.
    n = len(samples)
    results = {}

    # Test 1: Sample mean converges (LLN)
    expected_mean = samples.mean()
    batch_means = [samples[i:i+100].mean() for i in range(0, n-100, 100)]
    results['lln_convergence'] = np.std(batch_means) < 0.5

    # Test 2: No systematic time trends (stationarity)
    first_half = samples[:n//2].mean()
    second_half = samples[n//2:].mean()
    _, p_trend = stats.ttest_ind(samples[:n//2], samples[n//2:])
    results['no_trend'] = p_trend > 0.01

    # Test 3: Autocorrelation check
    lag1_corr = np.corrcoef(samples[:-1], samples[1:])[0,1]
    results['low_autocorr'] = abs(lag1_corr) < 0.05

    print(f"
📋 Distribution Property Tests ({name})")
    for test, passed in results.items():
        status = "✅" if passed else "❌"
        print(f"  {status} {test}")
    return all(results.values())

# Test good vs bad generators
good_samples = rng.normal(0, 1, 10000)
# Bad: trending data (non-stationary)
bad_samples  = np.arange(10000) * 0.001 + rng.normal(0, 1, 10000)

verify_distribution_properties(good_samples, "Good (Normal i.i.d.)")
verify_distribution_properties(bad_samples, "Bad (Trending data)")

In [ ]:
# ── Debugging a Specific Bug: Incorrect Sampling ──
print("🔍 Debugging Case Study: Deck Shuffling Bug")
print()

def buggy_shuffle(deck):
    # Bug: selects j from [0, n) not [i, n) -- not uniform!
    deck = deck.copy()
    n = len(deck)
    for i in range(n):
        j = rng.integers(0, n)  # BUG! Should be rng.integers(i, n)
        deck[i], deck[j] = deck[j], deck[i]
    return deck

def correct_shuffle(deck):
    deck = deck.copy()
    n = len(deck)
    for i in range(n-1, 0, -1):
        j = rng.integers(0, i+1)
        deck[i], deck[j] = deck[j], deck[i]
    return deck

# Test: what's P(card 0 ends up in position 0)?
deck = list(range(10))
n_trials = 100_000

buggy_pos0  = sum(buggy_shuffle(deck)[0] == 0 for _ in range(n_trials))
correct_pos0 = sum(correct_shuffle(deck)[0] == 0 for _ in range(n_trials))

expected = n_trials / 10  # should be 1/10 = 10%
print(f"P(card 0 stays in position 0):")
print(f"  Correct:  {correct_pos0/n_trials:.4f} (expected {1/10:.4f})")
print(f"  Buggy:    {buggy_pos0/n_trials:.4f} ← TOO HIGH! (bias toward original position)")
print()
print("💡 Statistical test caught the bug that visual inspection missed!")
print("   The buggy shuffle is ~17% likely to leave card 0 in place (should be 10%)")

## ✏️ Section 6 — Final Challenges (Track 2)

**C1:** Write a `assert_distribution` function that takes a list of samples and a `scipy.stats` distribution object, runs KS test, and passes/fails with a clear message.

**C2:** Test the Python `random.shuffle()` function for uniformity using chi-square on a small list [1,2,3].

**C3:** You have a caching layer that should hit the cache 80% of the time. Write a test that verifies the hit rate is statistically consistent with 80% using a binomial test.

<details><summary>Solutions</summary>See code below.</details>

In [ ]:
import random

# C1: assert_distribution
def assert_distribution(samples, expected_dist, alpha=0.05, name=""):
    ks_stat, p_value = stats.kstest(samples, expected_dist.cdf)
    passed = p_value > alpha
    msg = f"{'✅ PASS' if passed else '❌ FAIL'} [{name}]: KS={ks_stat:.4f}, p={p_value:.4f}"
    print(msg)
    if not passed:
        raise AssertionError(f"Distribution mismatch! {msg}")
    return True

samples = rng.normal(5, 2, 10000)
assert_distribution(samples, stats.norm(5, 2), name="Normal(5,2) test")

# C2: Test Python shuffle
from collections import Counter
import itertools

perms = Counter()
for _ in range(60000):
    lst = [1, 2, 3]
    random.shuffle(lst)
    perms[tuple(lst)] += 1

all_perms = list(itertools.permutations([1,2,3]))
counts = [perms[p] for p in all_perms]
expected = [10000] * 6
chi2, p = stats.chisquare(counts, expected)
print(f"
C2 Python shuffle chi2 test: χ²={chi2:.4f}, p={p:.4f}")
print("✅ Uniform" if p > 0.05 else "❌ Not uniform")

# C3: Cache hit rate test
n_requests = 1000
cache_hits = rng.binomial(n_requests, 0.80)
# Binomial test: H0: p=0.80
p_binom = stats.binom_test(cache_hits, n_requests, 0.80)
print(f"
C3 Cache hit test: {cache_hits}/{n_requests} = {cache_hits/n_requests:.2%}")
print(f"  p-value = {p_binom:.4f} → {'✅ Consistent with 80%' if p_binom > 0.05 else '❌ Not 80%'}")

## 🎯 Track 2 Complete! 🏆

**You've mastered:**
- ✅ PRNGs: seeds, reproducibility, quality tests
- ✅ Monte Carlo: integration, convergence, variance reduction
- ✅ A/B testing: design, analysis, peeking problem
- ✅ Bayesian updating: conjugate priors, Thompson Sampling
- ✅ Log probabilities: numerical stability, softmax, cross-entropy
- ✅ API probability: Poisson traffic, rate limiting, queuing theory
- ✅ Randomized algorithms: quicksort, reservoir sampling, hashing
- ✅ Probabilistic data structures: Bloom filters, Count-Min Sketch
- ✅ Feature flags: sequential testing, multi-armed bandits
- ✅ Debugging probabilistic systems: KS test, chi-square, property testing

**Ready for Tier 3?** → [Chapter 23 — Markov Chains]